In [ ]:
!pip3 install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

In [ ]:
categories = {
    "Mystery": "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html",
    "Historical Fiction": "https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html",
    "Romance": "https://books.toscrape.com/catalogue/category/books/romance_8/index.html"
}

books = []

In [ ]:
for category_name, category_url in categories.items():
    print(f"\nScraping category: {category_name}")
    current_url = category_url

    while current_url:

        response = requests.get(current_url)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Find all books on this page
        book_items = soup.select("article.product_pod")

        for book in soup.select("article.product_pod"):
            try:
                # Title
                title_element = book.select_one("h3 a")
                title = title_element.get("title", "").strip() if title_element else None

                # Price
                price_element = book.select_one(".price_color")
                price = price_element.get_text(strip=True) if price_element else None

                # Rating
                rating_element = book.select_one(".star-rating")

                if rating_element and len(rating_element.get("class", [])) > 1:
                    rating = rating_element.get("class")[1]
                else:
                    rating = None

                # Availability
                availability_element = book.select_one(".availability")

                if availability_element:
                    availability = availability_element.get_text(" ", strip=True)
                else:
                    availability = None

                # Store book
                books.append({
                    "title": title,
                    "price": price,
                    "star_rating": rating,
                    "availability": availability,
                    "category": category_name
                })

            except Exception as e:
                print(f"⚠️ Skipping book due to parsing error: {e}")
                continue

        # Find next page
        next_button = soup.select_one("li.next a")

        if next_button:
            next_url = next_button["href"]
            current_url = urljoin(current_url, next_url)
        else:
            current_url = None

df = pd.DataFrame(books)

print("\n--------------------------------")
print("Scraping completed")
print("--------------------------------")

print("Total books:", len(df))
print("Categories:", df["category"].nunique())

print("\nBooks per category:")
print(df["category"].value_counts())

print("\nFirst 5 rows:")
print(df.head())


Scraping category: Mystery

Scraping category: Historical Fiction

Scraping category: Romance

--------------------------------
Scraping completed
--------------------------------
Total books: 93
Categories: 3

Books per category:
category
Romance               35
Mystery               32
Historical Fiction    26
Name: count, dtype: int64

First 5 rows:
                                             title    price star_rating  \
0                                    Sharp Objects  Â£47.82        Four   
1                             In a Dark, Dark Wood  Â£19.63         One   
2                              The Past Never Ends  Â£56.50        Four   
3                                 A Murder in Time  Â£16.64         One   
4  The Murder of Roger Ackroyd (Hercule Poirot #4)  Â£44.10        Four   

  availability category  
0     In stock  Mystery  
1     In stock  Mystery  
2     In stock  Mystery  
3     In stock  Mystery  
4     In stock  Mystery  


In [ ]:
df.to_csv("raw_books.csv", index=False)

In [ ]:
print(df.shape)

(93, 5)


In [ ]:
print(df["category"].value_counts())

category
Romance               35
Mystery               32
Historical Fiction    26
Name: count, dtype: int64


In [ ]:
print(df.head(10))

                                             title    price star_rating  \
0                                    Sharp Objects  Â£47.82        Four   
1                             In a Dark, Dark Wood  Â£19.63         One   
2                              The Past Never Ends  Â£56.50        Four   
3                                 A Murder in Time  Â£16.64         One   
4  The Murder of Roger Ackroyd (Hercule Poirot #4)  Â£44.10        Four   
5                   The Last Mile (Amos Decker #2)  Â£54.21         Two   
6           That Darkness (Gardiner and Renner #1)  Â£13.92         One   
7             Tastes Like Fear (DI Marnie Rome #3)  Â£10.69         One   
8           A Time of Torment (Charlie Parker #14)  Â£48.35        Five   
9          A Study in Scarlet (Sherlock Holmes #1)  Â£16.73         Two   

  availability category  
0     In stock  Mystery  
1     In stock  Mystery  
2     In stock  Mystery  
3     In stock  Mystery  
4     In stock  Mystery  
5     In stock  My

In [ ]:
import pandas as pd

df = pd.read_csv("raw_books.csv")

print(df.head())

                                             title    price star_rating  \
0                                    Sharp Objects  Â£47.82        Four   
1                             In a Dark, Dark Wood  Â£19.63         One   
2                              The Past Never Ends  Â£56.50        Four   
3                                 A Murder in Time  Â£16.64         One   
4  The Murder of Roger Ackroyd (Hercule Poirot #4)  Â£44.10        Four   

  availability category  
0     In stock  Mystery  
1     In stock  Mystery  
2     In stock  Mystery  
3     In stock  Mystery  
4     In stock  Mystery  


In [ ]:
df

,title,price,star_rating,availability,category
0,Sharp Objects,Â£47.82,Four,In stock,Mystery
1,"In a Dark, Dark Wood",Â£19.63,One,In stock,Mystery
2,The Past Never Ends,Â£56.50,Four,In stock,Mystery
3,A Murder in Time,Â£16.64,One,In stock,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),Â£44.10,Four,In stock,Mystery
...,...,...,...,...,...
88,Imperfect Harmony,Â£34.74,Four,In stock,Romance
89,Fighting Fate (Fighting #6),Â£39.24,Three,In stock,Romance
90,Deep Under (Walker Security #1),Â£47.09,Five,In stock,Romance
91,Charity's Cross (Charles Towne Belles #4),Â£41.24,One,In stock,Romance


In [ ]:
df["price_gbp"] = (
    df["price"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)
    .astype(float)
)

In [ ]:
df

,title,price,star_rating,availability,category,price_gbp
0,Sharp Objects,Â£47.82,Four,In stock,Mystery,47.82
1,"In a Dark, Dark Wood",Â£19.63,One,In stock,Mystery,19.63
2,The Past Never Ends,Â£56.50,Four,In stock,Mystery,56.50
3,A Murder in Time,Â£16.64,One,In stock,Mystery,16.64
4,The Murder of Roger Ackroyd (Hercule Poirot #4),Â£44.10,Four,In stock,Mystery,44.10
...,...,...,...,...,...,...
88,Imperfect Harmony,Â£34.74,Four,In stock,Romance,34.74
89,Fighting Fate (Fighting #6),Â£39.24,Three,In stock,Romance,39.24
90,Deep Under (Walker Security #1),Â£47.09,Five,In stock,Romance,47.09
91,Charity's Cross (Charles Towne Belles #4),Â£41.24,One,In stock,Romance,41.24


In [ ]:
df = df.drop(columns=["price"])

In [ ]:
df

,title,star_rating,availability,category,price_gbp
0,Sharp Objects,Four,In stock,Mystery,47.82
1,"In a Dark, Dark Wood",One,In stock,Mystery,19.63
2,The Past Never Ends,Four,In stock,Mystery,56.50
3,A Murder in Time,One,In stock,Mystery,16.64
4,The Murder of Roger Ackroyd (Hercule Poirot #4),Four,In stock,Mystery,44.10
...,...,...,...,...,...
88,Imperfect Harmony,Four,In stock,Romance,34.74
89,Fighting Fate (Fighting #6),Three,In stock,Romance,39.24
90,Deep Under (Walker Security #1),Five,In stock,Romance,47.09
91,Charity's Cross (Charles Towne Belles #4),One,In stock,Romance,41.24


In [ ]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["star_rating"].map(rating_map)

In [ ]:
df["in_stock"] = df["availability"].str.contains(
    "In stock",
    case=False,
    na=False
)

In [ ]:
GBP_TO_INR = 105.50

df["price_inr"] = df["price_gbp"] * GBP_TO_INR

In [ ]:
print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)


Missing values:
title           0
price           0
star_rating     0
availability    0
category        0
price_gbp       0
rating          0
in_stock        0
price_inr       0
dtype: int64

Data types:
title            object
price            object
star_rating      object
availability     object
category         object
price_gbp       float64
rating            int64
in_stock           bool
price_inr       float64
dtype: object


In [ ]:
df = df[
    [
        "title",
        "price_gbp",
        "rating",
        "in_stock",
        "price_inr",
        "category"
    ]
]

In [ ]:
print(df.head())
print(df.shape)

                                             title  price_gbp  rating  \
0                                    Sharp Objects      47.82       4   
1                             In a Dark, Dark Wood      19.63       1   
2                              The Past Never Ends      56.50       4   
3                                 A Murder in Time      16.64       1   
4  The Murder of Roger Ackroyd (Hercule Poirot #4)      44.10       4   

   in_stock  price_inr category  
0      True   5045.010  Mystery  
1      True   2070.965  Mystery  
2      True   5960.750  Mystery  
3      True   1755.520  Mystery  
4      True   4652.550  Mystery  
(93, 6)


In [ ]:
df.to_csv("cleaned_books.csv", index=False)

In [ ]:
import sqlite3
import pandas as pd

In [ ]:
df = pd.read_csv("cleaned_books.csv")

print("Cleaned data loaded:")
print(df.head())

print("\nTotal books:", len(df))
print("Total categories:", df["category"].nunique())

Cleaned data loaded:
                                             title  price_gbp  rating  \
0                                    Sharp Objects      47.82       4   
1                             In a Dark, Dark Wood      19.63       1   
2                              The Past Never Ends      56.50       4   
3                                 A Murder in Time      16.64       1   
4  The Murder of Roger Ackroyd (Hercule Poirot #4)      44.10       4   

   in_stock  price_inr category  
0      True   5045.010  Mystery  
1      True   2070.965  Mystery  
2      True   5960.750  Mystery  
3      True   1755.520  Mystery  
4      True   4652.550  Mystery  

Total books: 93
Total categories: 3


In [ ]:
conn = sqlite3.connect("books.db")

cursor = conn.cursor()

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

In [ ]:
for category in df["category"].dropna().unique():

    cursor.execute("""
    INSERT OR IGNORE INTO categories (category_name)
    VALUES (?)
    """, (category,))

In [ ]:
for _, row in df.iterrows():

    # Get category ID
    cursor.execute("""
    SELECT category_id
    FROM categories
    WHERE category_name = ?
    """, (row["category"],))

    category_id = cursor.fetchone()[0]

    cursor.execute("""
    INSERT INTO books
    (
        title,
        price_gbp,
        price_inr,
        rating,
        in_stock,
        category_id
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        row["rating"],
        int(row["in_stock"]),
        category_id
    ))

In [ ]:
conn.commit()

In [ ]:
print("\nCategories:")
print(pd.read_sql(
    "SELECT * FROM categories",
    conn
))

print("\nBooks:")
print(pd.read_sql(
    "SELECT * FROM books LIMIT 10",
    conn
))


Categories:
   category_id       category_name
0            1             Mystery
1            2  Historical Fiction
2            3             Romance

Books:
   book_id                                            title  price_gbp  \
0        1                                    Sharp Objects      47.82   
1        2                             In a Dark, Dark Wood      19.63   
2        3                              The Past Never Ends      56.50   
3        4                                 A Murder in Time      16.64   
4        5  The Murder of Roger Ackroyd (Hercule Poirot #4)      44.10   
5        6                   The Last Mile (Amos Decker #2)      54.21   
6        7           That Darkness (Gardiner and Renner #1)      13.92   
7        8             Tastes Like Fear (DI Marnie Rome #3)      10.69   
8        9           A Time of Torment (Charlie Parker #14)      48.35   
9       10          A Study in Scarlet (Sherlock Holmes #1)      16.73   

   price_inr  rating  in

In [ ]:
conn.close()

print("Database created successfully!")

Database created successfully!


In [ ]:
import sqlite3

conn = sqlite3.connect("books.db")

query = """
SELECT
    books.title,
    books.rating,
    categories.category_name
FROM books
JOIN categories
ON books.category_id = categories.category_id
LIMIT 10;
"""

result = conn.execute(query)

for row in result:
    print(row)

('Sharp Objects', 4, 'Mystery')
('In a Dark, Dark Wood', 1, 'Mystery')
('The Past Never Ends', 4, 'Mystery')
('A Murder in Time', 1, 'Mystery')
('The Murder of Roger Ackroyd (Hercule Poirot #4)', 4, 'Mystery')
('The Last Mile (Amos Decker #2)', 2, 'Mystery')
('That Darkness (Gardiner and Renner #1)', 1, 'Mystery')
('Tastes Like Fear (DI Marnie Rome #3)', 1, 'Mystery')
('A Time of Torment (Charlie Parker #14)', 5, 'Mystery')
('A Study in Scarlet (Sherlock Holmes #1)', 2, 'Mystery')


In [ ]:
import sqlite3
import pandas as pd

# Connect to SQLite database
conn = sqlite3.connect("books.db")

print("✅ Connected to SQLite database")

✅ Connected to SQLite database


In [ ]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tables = pd.read_sql(query, conn)

tables

,name
0,categories
1,sqlite_sequence
2,books


In [ ]:
query = """
SELECT *
FROM books
LIMIT 10;
"""

books_sample = pd.read_sql(query, conn)

books_sample

,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,Sharp Objects,47.82,5045.010,4,1,1
1,2,"In a Dark, Dark Wood",19.63,2070.965,1,1,1
2,3,The Past Never Ends,56.50,5960.750,4,1,1
3,4,A Murder in Time,16.64,1755.520,1,1,1
4,5,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4652.550,4,1,1
5,6,The Last Mile (Amos Decker #2),54.21,5719.155,2,1,1
6,7,That Darkness (Gardiner and Renner #1),13.92,1468.560,1,1,1
7,8,Tastes Like Fear (DI Marnie Rome #3),10.69,1127.795,1,1,1
8,9,A Time of Torment (Charlie Parker #14),48.35,5100.925,5,1,1
9,10,A Study in Scarlet (Sherlock Holmes #1),16.73,1765.015,2,1,1


In [ ]:
query1 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating = 5;
"""

result1 = pd.read_sql(query1, conn)

result1

,title,price_gbp,rating
0,A Time of Torment (Charlie Parker #14),48.35,5
1,What Happened on Beale Street (Secrets of the ...,25.37,5
2,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5
3,The Silkworm (Cormoran Strike #2),23.05,5
4,The Girl You Lost,12.29,5
5,A Flight of Arrows (The Pathfinders #2),55.53,5
6,Mrs. Houdini,30.25,5
7,The Passion of Dolssa,28.32,5
8,Voyager (Outlander #3),21.07,5
9,The Red Tent,35.66,5


In [ ]:
query2 = """
SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC;
"""

result2 = pd.read_sql(query2, conn)

result2

,title,price_gbp
0,The Perfect Play (Play by Play #1),59.99
1,Boar Island (Anna Pigeon #19),59.48
2,Listen to Me (Fusion #1),58.99
3,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70
4,The Past Never Ends,56.50
...,...,...
88,The Girl You Lost,12.29
89,The Purest Hook (Second Circle Tattoos #3),12.25
90,Hide Away (Eve Duncan #20),11.84
91,Reservations for Two,11.10


In [ ]:
query3 = """
SELECT title, price_gbp
FROM books
ORDER BY price_gbp ASC
LIMIT 10;
"""

result3 = pd.read_sql(query3, conn)

result3

,title,price_gbp
0,Tastes Like Fear (DI Marnie Rome #3),10.69
1,Reservations for Two,11.10
2,Hide Away (Eve Duncan #20),11.84
3,The Purest Hook (Second Circle Tattoos #3),12.25
4,The Girl You Lost,12.29
5,Dark Lover (Black Dagger Brotherhood #1),12.87
6,Changing the Game (Play by Play #2),13.38
7,Playing with Fire,13.71
8,That Darkness (Gardiner and Renner #1),13.92
9,A Gentleman's Position (Society of Gentlemen #3),14.75


In [ ]:
query4 = """
SELECT DISTINCT category_id
FROM books;
"""

result4 = pd.read_sql(query4, conn)

result4

,category_id
0,1
1,2
2,3


In [ ]:
query5 = """
SELECT title, price_gbp, price_inr
FROM books
WHERE price_gbp BETWEEN 10 AND 20;
"""

result5 = pd.read_sql(query5, conn)

result5

,title,price_gbp,price_inr
0,"In a Dark, Dark Wood",19.63,2070.965
1,A Murder in Time,16.64,1755.520
2,That Darkness (Gardiner and Renner #1),13.92,1468.560
3,Tastes Like Fear (DI Marnie Rome #3),10.69,1127.795
4,A Study in Scarlet (Sherlock Holmes #1),16.73,1765.015
5,Hide Away (Eve Duncan #20),11.84,1249.120
6,Playing with Fire,13.71,1446.405
7,The Cuckoo's Calling (Cormoran Strike #1),19.21,2026.655
8,The Girl You Lost,12.29,1296.595
9,The Girl In The Ice (DCI Erika Foster #1),15.85,1672.175


In [ ]:
query6 = """
SELECT
    books.title,
    books.price_gbp,
    books.rating,
    categories.category_name
FROM books
JOIN categories
    ON books.category_id = categories.category_id;
"""

result6 = pd.read_sql(query6, conn)

result6.head(10)

,title,price_gbp,rating,category_name
0,Sharp Objects,47.82,4,Mystery
1,"In a Dark, Dark Wood",19.63,1,Mystery
2,The Past Never Ends,56.50,4,Mystery
3,A Murder in Time,16.64,1,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4,Mystery
5,The Last Mile (Amos Decker #2),54.21,2,Mystery
6,That Darkness (Gardiner and Renner #1),13.92,1,Mystery
7,Tastes Like Fear (DI Marnie Rome #3),10.69,1,Mystery
8,A Time of Torment (Charlie Parker #14),48.35,5,Mystery
9,A Study in Scarlet (Sherlock Holmes #1),16.73,2,Mystery


In [ ]:
query7 = """
SELECT
    categories.category_name,
    COUNT(books.book_id) AS total_books
FROM categories
JOIN books
    ON categories.category_id = books.category_id
GROUP BY categories.category_name
ORDER BY total_books DESC;
"""

result7 = pd.read_sql(query7, conn)

result7

,category_name,total_books
0,Romance,35
1,Mystery,32
2,Historical Fiction,26


In [ ]:
print("QUERY 1 — 5 STAR BOOKS")
display(result1)

print("QUERY 2 — MOST EXPENSIVE BOOKS")
display(result2.head(10))

print("QUERY 3 — 10 CHEAPEST BOOKS")
display(result3)

print("QUERY 4 — DISTINCT CATEGORIES")
display(result4)

print("QUERY 5 — BOOKS BETWEEN £10 AND £20")
display(result5)

print("QUERY 6 — BOOK + CATEGORY JOIN")
display(result6.head(10))

print("QUERY 7 — BOOK COUNT BY CATEGORY")
display(result7)

QUERY 1 — 5 STAR BOOKS


,title,price_gbp,rating
0,A Time of Torment (Charlie Parker #14),48.35,5
1,What Happened on Beale Street (Secrets of the ...,25.37,5
2,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5
3,The Silkworm (Cormoran Strike #2),23.05,5
4,The Girl You Lost,12.29,5
5,A Flight of Arrows (The Pathfinders #2),55.53,5
6,Mrs. Houdini,30.25,5
7,The Passion of Dolssa,28.32,5
8,Voyager (Outlander #3),21.07,5
9,The Red Tent,35.66,5


QUERY 2 — MOST EXPENSIVE BOOKS


,title,price_gbp
0,The Perfect Play (Play by Play #1),59.99
1,Boar Island (Anna Pigeon #19),59.48
2,Listen to Me (Fusion #1),58.99
3,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70
4,The Past Never Ends,56.50
5,A Walk to Remember,56.43
6,Suddenly in Love (Lake Haven #1),55.99
7,The Last Painting of Sara de Vos,55.55
8,A Flight of Arrows (The Pathfinders #2),55.53
9,Murder at the 42nd Street Library (Raymond Amb...,54.36


QUERY 3 — 10 CHEAPEST BOOKS


...

,title,price_gbp,price_inr
0,"In a Dark, Dark Wood",19.63,2070.965
1,A Murder in Time,16.64,1755.520
2,That Darkness (Gardiner and Renner #1),13.92,1468.560
3,Tastes Like Fear (DI Marnie Rome #3),10.69,1127.795
4,A Study in Scarlet (Sherlock Holmes #1),16.73,1765.015
5,Hide Away (Eve Duncan #20),11.84,1249.120
6,Playing with Fire,13.71,1446.405
7,The Cuckoo's Calling (Cormoran Strike #1),19.21,2026.655
8,The Girl You Lost,12.29,1296.595
9,The Girl In The Ice (DCI Erika Foster #1),15.85,1672.175


QUERY 6 — BOOK + CATEGORY JOIN


,title,price_gbp,rating,category_name
0,Sharp Objects,47.82,4,Mystery
1,"In a Dark, Dark Wood",19.63,1,Mystery
2,The Past Never Ends,56.50,4,Mystery
3,A Murder in Time,16.64,1,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4,Mystery
5,The Last Mile (Amos Decker #2),54.21,2,Mystery
6,That Darkness (Gardiner and Renner #1),13.92,1,Mystery
7,Tastes Like Fear (DI Marnie Rome #3),10.69,1,Mystery
8,A Time of Torment (Charlie Parker #14),48.35,5,Mystery
9,A Study in Scarlet (Sherlock Holmes #1),16.73,2,Mystery


QUERY 7 — BOOK COUNT BY CATEGORY


,category_name,total_books
0,Romance,35
1,Mystery,32
2,Historical Fiction,26


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("books.db")

print("✅ Connected to SQLite")

✅ Connected to SQLite


In [ ]:
query = """
SELECT title, price_gbp, price_inr, rating
FROM books
WHERE rating = 5;
"""

df_rating = pd.read_sql(query, conn)

df_rating

,title,price_gbp,price_inr,rating
0,A Time of Torment (Charlie Parker #14),48.35,5100.925,5
1,What Happened on Beale Street (Secrets of the ...,25.37,2676.535,5
2,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5517.650,5
3,The Silkworm (Cormoran Strike #2),23.05,2431.775,5
4,The Girl You Lost,12.29,1296.595,5
5,A Flight of Arrows (The Pathfinders #2),55.53,5858.415,5
6,Mrs. Houdini,30.25,3191.375,5
7,The Passion of Dolssa,28.32,2987.760,5
8,Voyager (Outlander #3),21.07,2222.885,5
9,The Red Tent,35.66,3762.130,5


In [ ]:
query = """
SELECT title, price_gbp, price_inr, rating
FROM books
WHERE price_gbp BETWEEN 10 AND 20;
"""

df_price = pd.read_sql(query, conn)

df_price

,title,price_gbp,price_inr,rating
0,"In a Dark, Dark Wood",19.63,2070.965,1
1,A Murder in Time,16.64,1755.520,1
2,That Darkness (Gardiner and Renner #1),13.92,1468.560,1
3,Tastes Like Fear (DI Marnie Rome #3),10.69,1127.795,1
4,A Study in Scarlet (Sherlock Holmes #1),16.73,1765.015,2
5,Hide Away (Eve Duncan #20),11.84,1249.120,1
6,Playing with Fire,13.71,1446.405,3
7,The Cuckoo's Calling (Cormoran Strike #1),19.21,2026.655,1
8,The Girl You Lost,12.29,1296.595,5
9,The Girl In The Ice (DCI Erika Foster #1),15.85,1672.175,3


In [ ]:
print("5-Star Books:")
print(df_rating.shape)

print("\nBooks between £10 and £20:")
print(df_price.shape)

5-Star Books:
(17, 4)

Books between £10 and £20:
(20, 4)


In [ ]:
books_df = pd.read_sql("SELECT * FROM books", conn)

categories_df = pd.read_sql("SELECT * FROM categories", conn)

print("Books:")
display(books_df.head())

print("Categories:")
display(categories_df.head())

Books:


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,Sharp Objects,47.82,5045.010,4,1,1
1,2,"In a Dark, Dark Wood",19.63,2070.965,1,1,1
2,3,The Past Never Ends,56.50,5960.750,4,1,1
3,4,A Murder in Time,16.64,1755.520,1,1,1
4,5,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4652.550,4,1,1


Categories:


,category_id,category_name
0,1,Mystery
1,2,Historical Fiction
2,3,Romance


In [ ]:
merged_df = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

merged_df.head(10)

,book_id,title,price_gbp,price_inr,rating,in_stock,category_id,category_name
0,1,Sharp Objects,47.82,5045.010,4,1,1,Mystery
1,2,"In a Dark, Dark Wood",19.63,2070.965,1,1,1,Mystery
2,3,The Past Never Ends,56.50,5960.750,4,1,1,Mystery
3,4,A Murder in Time,16.64,1755.520,1,1,1,Mystery
4,5,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4652.550,4,1,1,Mystery
5,6,The Last Mile (Amos Decker #2),54.21,5719.155,2,1,1,Mystery
6,7,That Darkness (Gardiner and Renner #1),13.92,1468.560,1,1,1,Mystery
7,8,Tastes Like Fear (DI Marnie Rome #3),10.69,1127.795,1,1,1,Mystery
8,9,A Time of Torment (Charlie Parker #14),48.35,5100.925,5,1,1,Mystery
9,10,A Study in Scarlet (Sherlock Holmes #1),16.73,1765.015,2,1,1,Mystery


In [ ]:
sql_join = """
SELECT
    books.title,
    books.price_gbp,
    books.rating,
    categories.category_name
FROM books
JOIN categories
    ON books.category_id = categories.category_id;
"""

sql_result = pd.read_sql(sql_join, conn)

sql_result.head()

,title,price_gbp,rating,category_name
0,Sharp Objects,47.82,4,Mystery
1,"In a Dark, Dark Wood",19.63,1,Mystery
2,The Past Never Ends,56.50,4,Mystery
3,A Murder in Time,16.64,1,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4,Mystery


In [ ]:
pandas_result = merged_df[
    ["title", "price_gbp", "rating", "category_name"]
]

pandas_result.head()

,title,price_gbp,rating,category_name
0,Sharp Objects,47.82,4,Mystery
1,"In a Dark, Dark Wood",19.63,1,Mystery
2,The Past Never Ends,56.50,4,Mystery
3,A Murder in Time,16.64,1,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4,Mystery


In [ ]:
print("SQL JOIN rows:", len(sql_result))
print("Pandas merge rows:", len(pandas_result))

SQL JOIN rows: 93
Pandas merge rows: 93


In [ ]:
sql_check = sql_result.sort_values(
    ["title", "category_name"]
).reset_index(drop=True)

pandas_check = pandas_result.sort_values(
    ["title", "category_name"]
).reset_index(drop=True)

print(
    "SQL JOIN and Pandas merge equivalent:",
    sql_check.equals(pandas_check)
)

SQL JOIN and Pandas merge equivalent: True


ReadME

In [ ]:
readme_content = """
# Zepto Data & AI Platform — Module 1

## 1. Project Overview

Module 1 focuses on building a data pipeline using web scraping,
data cleaning, SQLite database design, SQL queries, and Pandas.

The pipeline follows:

Scraping → Cleaning → Currency Conversion → SQLite → SQL → Pandas

---

## 2. Data Source

Data was collected from:

Books to Scrape
https://books.toscrape.com/

The website is used for scraping practice and does not require
login credentials or an API key.

---

## 3. Technologies Used

- Python
- Requests
- BeautifulSoup
- Pandas
- SQLite
- Google Colab

---

## 4. Data Collection

The scraper automatically collected book information from
multiple categories.

The following fields were collected:

- Title
- Price
- Star Rating
- Availability
- Category

At least 60 books were collected from at least 3 categories.

---

## 5. Data Cleaning

The scraped data was cleaned before storing it in SQLite.

### Price

The original price was converted from text to float.

Example:

£51.77 → 51.77

### Rating

Star ratings were converted to integers:

One → 1
Two → 2
Three → 3
Four → 4
Five → 5

### Availability

Availability was converted into a Boolean value:

In stock → True
Not in stock → False

---

## 6. Currency Conversion

A fixed conversion rate was used:

1 GBP = 105.50 INR

Formula:

price_inr = price_gbp × 105.50

---

## 7. SQLite Database

The cleaned data was stored in SQLite.

Two related tables were created:

### categories

- category_id
- category_name

### books

- book_id
- title
- price_gbp
- price_inr
- rating
- in_stock
- category_id

The `category_id` connects the books table with the categories table.

---

## 8. SQL Queries

The project contains SQL queries demonstrating:

1. SELECT and WHERE
2. ORDER BY
3. LIMIT
4. DISTINCT
5. BETWEEN
6. JOIN

The SQL queries were executed using SQLite and their outputs
were displayed in the notebook.

---

## 9. Pandas read_sql()

SQL query results were loaded into Pandas DataFrames using:

pd.read_sql()

At least two SQL query results were loaded into Pandas.

---

## 10. Pandas Merge

The relationship between books and categories was reproduced
using:

pd.merge()

The Pandas merge result was compared with the SQL JOIN result
to verify that both approaches produced equivalent results.

---

## 11. Project Workflow

Scraping
↓
Cleaning
↓
Currency Conversion
↓
SQLite Database
↓
SQL Queries
↓
Pandas read_sql
↓
Pandas merge

---

## 12. Conclusion

Module 1 demonstrates an end-to-end data pipeline starting from
web scraping and ending with SQL and Pandas analysis.

The pipeline provides the data foundation required for the
next modules of the Zepto Data & AI Platform.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("✅ README.md created successfully")

✅ README.md created successfully


In [ ]:
with open("README.md", "r", encoding="utf-8") as f:
    print(f.read())


# Zepto Data & AI Platform — Module 1

## 1. Project Overview

Module 1 focuses on building a data pipeline using web scraping,
data cleaning, SQLite database design, SQL queries, and Pandas.

The pipeline follows:

Scraping → Cleaning → Currency Conversion → SQLite → SQL → Pandas

---

## 2. Data Source

Data was collected from:

Books to Scrape
https://books.toscrape.com/

The website is used for scraping practice and does not require
login credentials or an API key.

---

## 3. Technologies Used

- Python
- Requests
- BeautifulSoup
- Pandas
- SQLite
- Google Colab

---

## 4. Data Collection

The scraper automatically collected book information from
multiple categories.

The following fields were collected:

- Title
- Price
- Star Rating
- Availability
- Category

At least 60 books were collected from at least 3 categories.

---

## 5. Data Cleaning

The scraped data was cleaned before storing it in SQLite.

### Price

The original price was converted from text to float.

Example:


In [ ]:
import os

folders = [
    "Zepto-Data-AI-Platform/Module-1-Data-Pipeline",
    "Zepto-Data-AI-Platform/Module-2-ML",
    "Zepto-Data-AI-Platform/Module-3-GenAI"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Project folders created")

✅ Project folders created


In [ ]:
print("========== MODULE 1 FINAL VALIDATION ==========")

print("Total books:", len(df))
print("Total categories:", df["category"].nunique())

print("\nCategories:")
print(df["category"].unique())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDatabase tables:")
print(pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
AND name NOT LIKE 'sqlite_%';
""", conn))

print("\nSQL JOIN rows:", len(sql_result))
print("Pandas merge rows:", len(pandas_result))

print(
    "\nSQL JOIN = Pandas merge:",
    sql_check.equals(pandas_check)
)

print("\n========== MODULE 1 COMPLETE ==========")

========== MODULE 1 FINAL VALIDATION ==========
Total books: 93
Total categories: 3

Categories:
['Mystery' 'Historical Fiction' 'Romance']

Missing values:
title           0
star_rating     0
availability    0
category        0
price_gbp       0
dtype: int64

Duplicate rows: 0

Database tables:
         name
0  categories
1       books

SQL JOIN rows: 93
Pandas merge rows: 93

SQL JOIN = Pandas merge: True

========== MODULE 1 COMPLETE ==========


In [76]:
from google.colab import files

files.download("/content/Zepto-Data-AI-Platform/Module-1-Data-Pipeline/cleaned_books.csv")
files.download("/content/Zepto-Data-AI-Platform/Module-1-Data-Pipeline/books.db")
files.download("/content/Zepto-Data-AI-Platform/Module-1-Data-Pipeline/README.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>